In [1]:
from steer_core.DataManager import DataManager

from steer_materials.CellMaterials.Base import CurrentCollectorMaterial, InsulationMaterial, SeparatorMaterial
from steer_materials.CellMaterials.Electrode import CathodeMaterial, Binder, ConductiveAdditive, AnodeMaterial

from steer_opencell_design.Components.CurrentCollectors import TablessCurrentCollector
from steer_opencell_design.Components.Electrodes import Cathode, Anode
from steer_opencell_design.Components.Separators import Separator
from steer_opencell_design.Constructions.Layups import Laminate
from steer_opencell_design.Formulations.ElectrodeFormulations import CathodeFormulation, AnodeFormulation
from steer_opencell_design.Constructions.ElectrodeAssemblies.JellyRolls import WoundJellyRoll
from steer_opencell_design.AuxillaryComponents.WindingEquipment import RoundMandrel

from steer_opencell_design import __version__

import pandas as pd
from datetime import datetime as dt

In [2]:
db = DataManager()

In [3]:
db.create_table(
    table_name='cells',
    columns={
        'name': 'TEXT PRIMARY KEY',
        'object': 'BLOB',
        'form_factor': 'TEXT',
        'internal_construction': 'TEXT',
        'date': 'TEXT',
        'version': 'TEXT',
        'reference': 'TEXT'
    }
)

In [4]:
db.remove_data(table_name='cells', condition="name = 'CBAK-32140NS'")

In [5]:
db.get_table_names()

['cells',
 'anode_materials',
 'binder_materials',
 'cathode_materials',
 'conductive_additive_materials',
 'current_collector_materials',
 'insulation_materials',
 'separator_materials']

In [6]:
# set some standard materials

conductive_additive = ConductiveAdditive.from_database("Super P")
binder = Binder.from_database("PVDF")
insulation = InsulationMaterial.from_database("Aluminium Oxide, 95%")
current_collector_material = CurrentCollectorMaterial.from_database('Aluminum')
separator_material = SeparatorMaterial.from_database('Polyethylene')



In [7]:
# Create the cathode

cathode_current_collector = TablessCurrentCollector(
    material=current_collector_material,
    width=130,
    length=3200,
    coated_width=125,
    insulation_width=2.5,
    thickness=13.5
)

cathode_active_material = CathodeMaterial.from_database("NFM111 (Vendor C)")

cathode_formulation = CathodeFormulation(
    active_materials={cathode_active_material: 95},
    binders={binder: 2.5},
    conductive_additives={conductive_additive: 2.5}
)

my_cathode = Cathode(
    formulation=cathode_formulation,
    current_collector=cathode_current_collector,
    calender_density=2.93,
    mass_loading=11.27,
    insulation_material=insulation,
    insulation_thickness=3
)

my_cathode.properties

{'Cost': '$ 1.17',
 'Mass': '103.7 g',
 'Coating mass': '88.36 g',
 'Total thickness': '90.4 um',
 'Coating thickness': '38.46 um'}

In [8]:
# Create the anode

anode_current_collector = TablessCurrentCollector(
    material=current_collector_material,
    width=133,
    length=3250,
    coated_width=128,
    insulation_width=2.5,
    thickness=13.5,
)

anode_active_material = AnodeMaterial.from_database("Hard Carbon (Vendor A)")

anode_formulation = AnodeFormulation(
    active_materials={anode_active_material: 95},
    binders={binder: 2.5},
    conductive_additives={conductive_additive: 2.5}
)

my_anode = Anode(
    formulation=anode_formulation,
    current_collector=anode_current_collector,
    calender_density=1.1,
    mass_loading=8,
    insulation_material=insulation,
    insulation_thickness=3
)

my_anode.properties

{'Cost': '$ 0.67',
 'Mass': '81.19 g',
 'Coating mass': '65.26 g',
 'Total thickness': '159.0 um',
 'Coating thickness': '72.73 um'}

In [9]:
# create the layup 

top_separator = Separator(
    material=separator_material, 
    thickness=12,
    width = 130,
    length = 3600
)

bottom_serparator = Separator(
    material=separator_material, 
    thickness=12,
    width = 130,
    length = 3600,
)

my_layup = Laminate(
    anode=my_anode,
    cathode=my_cathode,
    top_separator=top_separator,
    bottom_separator=bottom_serparator,
    name="CBAK-32140NS"
)



In [10]:
# create the jellyroll assembly

mandrel = RoundMandrel(
    diameter=5,
    length=500,
)

my_jellyroll = WoundJellyRoll(
    laminate=my_layup,
    mandrel=mandrel,
    name="CBAK-32140NS"
)


In [11]:
my_jellyroll.roll_properties

,Turns
Component,
anode_a_side_coating_turns,52.50
anode_current_collector_turns,52.50
anode_b_side_coating_turns,52.50
cathode_a_side_coating_turns,50.88
cathode_current_collector_turns,50.88
cathode_b_side_coating_turns,50.88
bottom_separator_turns,64.87
bottom_separator_inner_turns,10.69
bottom_separator_outer_turns,1.65


In [12]:
my_jellyroll.get_spiral_plot(height=1300, width=1300).show()

In [13]:

db.insert_data(table_name='cells', data=pd.DataFrame({
    'name': [my_jellyroll.name],
    'object': [my_jellyroll.serialize()],
    'form_factor': ['Cylindrical'],
    'internal_construction': ['Wound'],
    'date': [dt.now().strftime("%Y-%m-%d %H:%M:%S")],
    'version': [__version__],
    'reference': ['Na/Na+']
}))


In [16]:
db.get_data('cells')

,name,object,form_factor,internal_construction,date,version,reference
0,Cell 4,gASVuAABAAAAAACMKnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-10-21 15:18:36,0.3.12,Na/Na+
1,HiNa-NaCR32140-MP10,gASVswMAAAAAAACMKnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-10-21 15:18:38,0.3.12,Na/Na+
2,QNAS-S40160NL,gASVswMAAAAAAACMKnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-10-21 15:18:39,0.3.12,Na/Na+
3,QNAS-S40160RL,gASVswMAAAAAAACMKnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-10-21 15:18:41,0.3.12,Na/Na+
4,UniGrid-NCO32140,gASVzcIAAAAAAACMKnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-10-21 15:18:43,0.3.12,Na/Na+
5,Vendor D NFPP,gASVF/0AAAAAAACMKnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Prismatic,Stacked,2025-10-21 15:18:45,0.3.12,Na/Na+
6,Vendor E NFPP,gASVF/0AAAAAAACMKnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Prismatic,Stacked,2025-10-21 15:18:46,0.3.12,Na/Na+
7,Vendor G NFM,gASVswMAAAAAAACMKnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-10-21 15:18:48,0.3.12,Na/Na+
8,Vendor G NFPP,gASVaPQAAAAAAACMKnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-10-21 15:18:50,0.3.12,Na/Na+
9,CBAK-32140NS,gASVOAwAAAAAAACMQnN0ZWVyX29wZW5jZWxsX2Rlc2lnbi...,Cylindrical,Wound,2025-11-03 16:48:50,0.3.12,Na/Na+
